<a href="https://colab.research.google.com/github/simonaoliveto83/pw18-ticketing/blob/main/prototipo2/dashboard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [25]:
#from google.colab import drive
#drive.mount("/content/drive")

In [26]:
import os

if not os.path.exists("pw18-ticketing"):
    !git clone https://github.com/simonaoliveto83/pw18-ticketing.git

In [27]:
import torch
import torch.nn as nn
import pandas as pd
import joblib
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output

In [28]:
#BASE_PATH = "/content/drive/MyDrive/prototipo2/models"
BASE_PATH = "pw18-ticketing/prototipo2/models"

CATEGORY_MODEL_PATH = BASE_PATH + "/category_model.pt"
VECTORIZER_PATH = BASE_PATH + "/tfidf_vectorizer.joblib"
PRIORITY_MODEL_PATH = BASE_PATH + "/priority_model.pt"
VECTORIZER_PRIO_PATH = BASE_PATH + "/tfidf_prio_vectorizer.joblib"

In [29]:
class TicketClassifier(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        return self.net(x)

In [30]:
class TicketPriorityClassifier(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        return self.net(x)

In [31]:
vectorizer = joblib.load(VECTORIZER_PATH)
cat_input_dim = len(vectorizer.get_feature_names_out())
vectorizerPrio = joblib.load(VECTORIZER_PRIO_PATH)
prio_input_dim = len(vectorizerPrio.get_feature_names_out())

In [32]:
cat_model = TicketClassifier(cat_input_dim, 3)
prio_model = TicketPriorityClassifier(prio_input_dim, 3)

cat_model.load_state_dict(torch.load(CATEGORY_MODEL_PATH, map_location="cpu"))
prio_model.load_state_dict(torch.load(PRIORITY_MODEL_PATH, map_location="cpu"))

cat_model.eval()
prio_model.eval()

TicketPriorityClassifier(
  (net): Sequential(
    (0): Linear(in_features=1358, out_features=256, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=256, out_features=128, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.3, inplace=False)
    (6): Linear(in_features=128, out_features=3, bias=True)
  )
)

In [33]:
category_labels = {
    0: "Amministrazione",
    1: "Tecnico",
    2: "Commerciale"
}

priority_labels = {
    0: "Bassa",
    1: "Media",
    2: "Alta"
}


In [34]:
def top_words(vec, model, vectorizer, n=5):
    # mi assicuro che sia numpy array
    if hasattr(vec, "toarray"):
        vec = vec.toarray()
    vec = vec.flatten()

    # usa il primo layer linear
    first_layer = model.net[0]
    weights = first_layer.weight.detach().numpy()

    features = vectorizer.get_feature_names_out()
    idx = np.where(vec > 0)[0]  # indici TF-IDF attivi

    # sommo i pesi dei neuroni nascosti attivi
    scores = np.abs(weights[:, idx]).sum(axis=0)

    # prendo i top n
    top_idx = np.argsort(np.abs(scores))[-n:]
    return [features[idx[i]] for i in top_idx]

In [35]:
def predict_ticket(b):
    clear_output(wait=True)

    text = subject_input.value + " " + description_input.value

    if len(text.strip()) == 0:
        print("⚠️ Inserisci un testo")
        return

    vec_cat = vectorizer.transform([text])
    vec_cat_tensor = torch.tensor(vec_cat.toarray(), dtype=torch.float32)

    vec_prio = vectorizerPrio.transform([text])
    vec_prio_tensor = torch.tensor(vec_prio.toarray(), dtype=torch.float32)

    with torch.no_grad():
        cat_pred = torch.argmax(cat_model(vec_cat_tensor), dim=1).item()
        prio_pred = torch.argmax(prio_model(vec_prio_tensor), dim=1).item()

    print("📌 RISULTATO")
    print("Ticket:", subject_input.value, "-" , description_input.value)
    print("Categoria:", category_labels[cat_pred])
    print("Priorità sufferita:", priority_labels[prio_pred])
    print("Parole influenti:", ", ".join(top_words(vec_cat, cat_model, vectorizer)))

In [37]:
subject_input = widgets.Text(
    description="Oggetto:",
    layout=widgets.Layout(width="80%")
)

description_input = widgets.Textarea(
    description="Descrizione:",
    layout=widgets.Layout(width="80%", height="100px")
)

predict_button = widgets.Button(
    description="Classifica Ticket",
    button_style="primary"
)

predict_button.on_click(predict_ticket)

display(subject_input, description_input, predict_button)

📌 RISULTATO
Ticket: accoglienza - verificare che la sala attesa sia pulita
Categoria: Tecnico
Priorità sufferita: Bassa
Parole influenti: che, verificare, accoglienza, la, sala
